In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [21]:
from langchain.mcp import MCPAdapter
from fastmcp import Client
from langchain.agents import create_agent
from langchain.messages import HumanMessage

In [3]:
from langchain_groq import ChatGroq
llm = ChatGroq(model='openai/gpt-oss-20b',max_tokens=200,temperature=0.1)

In [11]:
client = Client('http://localhost:8000/mcp')
print(client)

In [19]:
async with MCPAdapter(client) as adapter:
    tools = await adapter.list_tools()
    print(tools)

[StructuredTool(name='get_conversion_factor', description='Get the currency conversion rate between a base currency and target currency.\nUse this tool first when a user asks to convert currencies. Pass its\nconversion rate result to the convert tool.', args_schema={'type': 'object', 'additionalProperties': False, 'properties': {'base_currency': {'type': 'string'}, 'target_currency': {'type': 'string'}}, 'required': ['base_currency', 'target_currency']}, metadata={'mcp': {'tool': {'_meta': {'fastmcp': {'tags': []}}}, 'server': {'name': 'Exchange Rate Conversion', 'version': '4.0.5'}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x7fa18a47afb0>, response_format='content_and_artifact', coroutine=<function as_langchain_tool.<locals>.call_tool at 0x7fa17f1db5e0>), StructuredTool(name='convert', description='Convert a currency amount using a conversion rate obtained from\nget_conversion_factor. Use this tool after getting the conversion rate.', args_schema={'type': 'object', 'add

In [24]:
async with MCPAdapter(client) as adapter:
    tools = await adapter.list_tools()
    
    agent = create_agent(
        model=llm,
        tools=tools,
    )

    query = HumanMessage('What is 20 dollars equivalent to indian rupee?')

    initial_state = {"messages": [query]}
    result = await agent.ainvoke(initial_state)
    print(result)

{'messages': [HumanMessage(content='What is 20 dollars equivalent to indian rupee?', additional_kwargs={}, response_metadata={}, id='737c8dec-395d-4cad-9fc7-a629c01252e1'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to convert USD to INR. Use get_conversion_factor tool.', 'tool_calls': [{'id': 'fc_191379f9-9273-42ba-85c9-9de9384f496f', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 226, 'total_tokens': 272, 'completion_time': 0.050059493, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.014089303, 'prompt_tokens_details': None, 'queue_time': 0.314558625, 'total_time': 0.064148796}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9340e7d14d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0

In [26]:
async with MCPAdapter(client) as adapter:
    tools = await adapter.list_tools()
    
    agent = create_agent(
        model=llm,
        tools=tools,
    )

    query = HumanMessage('What is 20 dollars equivalent to indian rupee?')

    initial_state = {"messages": [query]}
    result = await agent.ainvoke(initial_state)
    
    last_message = result["messages"][-1]
    print(last_message.content)

20 USD is roughly **₹1,919** (about ₹1,918.99).
